# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import certifi
import duckdb
from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

In [2]:
import re

_con = None

def get_duckdb_connection():
    global _con
    if _con is not None:
        return _con

    _con = duckdb.connect()
    _con.execute("INSTALL delta;")
    _con.execute("LOAD delta;")

    if str(DATA_DIR).startswith('s3://'):
        _con.execute("INSTALL httpfs;")
        _con.execute("LOAD httpfs;")

        missing = [name for name in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"] if not os.getenv(name)]
        if missing:
            raise RuntimeError(f"Missing AWS credential environment variable(s): {', '.join(missing)}")

        _con.execute(
            "CREATE OR REPLACE SECRET emaps_s3 (TYPE S3, KEY_ID ?, SECRET ?, REGION ?)",
            [
                os.environ["AWS_ACCESS_KEY_ID"],
                os.environ["AWS_SECRET_ACCESS_KEY"],
                os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1",
            ],
        )

    return _con

def print_sql_df(sql_query):
    con = get_duckdb_connection()
    
    # Automatically replace schema.table with delta_scan('{DATA_DIR}/schema/table')
    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)', 
        f'\\1 delta_scan(\'{DATA_DIR}/\\2/\\3\')', 
        sql_query
    )
    df = con.execute(parsed_query).df()
    display(df)

In [3]:
print("Gold Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix limit 10')

Gold Daily Mix Summary:


IOException: IO Error: DeltaKernel GenericError (5): Generic delta kernel error: No files in log segment

LINE 1: select * from delta_scan('s3://electricity-maps/data/bronze/electricity_mix...
                      ^

In [ ]:
print("Gold Daily Mix Summary:")
print_sql_df('select * from gold.daily_exports limit 10')

In [ ]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports LIMIT 10")

In [ ]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")